# 03 · Modeling & Honest Validation

The core of the project: baselines vs. the gradient-boosted risk model, evaluated
with **spatially and temporally honest** cross-validation, plus SHAP to confirm
the model learns fire physics.

> Run `uv run python scripts/04_train_tabular.py` first (it writes the metrics this
> notebook reads). Optionally tune with `scripts/08_tune.py`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from wildfire.config import load_config
cfg = load_config()
metrics = json.loads((cfg.path_for('metrics') / 'tabular_metrics.json').read_text())

rows = []
for scheme, models in metrics['schemes'].items():
    for name, res in models.items():
        a = res['aggregate']
        rows.append(dict(scheme=scheme, model=name,
                         pr_auc=a.get('pr_auc_mean'), recall_at_p20=a.get('recall_at_p20_mean'),
                         brier=a.get('brier_mean'), roc_auc=a.get('roc_auc_mean')))
cmp = pd.DataFrame(rows)
cmp.round(3)

## Why validation scheme matters

The climatology baseline (predict where fires historically recur) can look OK under
time-based CV but **collapses toward no-skill under spatial-block CV** — because it
has no skill on regions it never trained on. Random splits hide exactly this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, scheme in zip(axes, cmp['scheme'].unique()):
    sub = cmp[cmp['scheme'] == scheme].set_index('model')['pr_auc']
    colors = ['#1f6feb' if m == 'risk_gbm' else '#bbb' for m in sub.index]
    sub.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(f'PR-AUC · {scheme}'); ax.set_xlabel(''); ax.axhline(0, color='k', lw=0.5)
plt.tight_layout()

In [ ]:
# Calibration matters too: lower Brier = better-calibrated probabilities.
piv = cmp.pivot(index='model', columns='scheme', values='brier')
piv.plot(kind='bar', figsize=(9, 4)); plt.ylabel('Brier (lower better)')
plt.title('Probability calibration by model & CV scheme'); plt.tight_layout()

## SHAP — does the model learn fire physics?

If the top drivers are weather/fuel/drought (not location artifacts), we trust the
model for the right reasons.

In [ ]:
from wildfire.models.risk import RiskModel
from wildfire.eval.shap_explain import explain_risk
from wildfire.features.build import feature_columns

rm = RiskModel.load(cfg.path_for('models') / 'risk_model.joblib')
df = pd.read_parquet(cfg.path_for('data_processed') / 'features.parquet')
imp = explain_risk(rm, df, sample_n=4000, seed=cfg.seed)
imp.head(15).iloc[::-1].plot.barh(x='feature', y='mean_abs_shap', figsize=(8, 7),
                                  color='#c1440e', legend=False)
plt.xlabel('mean |SHAP|'); plt.title('Risk model drivers'); plt.tight_layout()

## Fire-count model (per ecoregion / season)

Negative-binomial counts with calibrated prediction intervals.

In [ ]:
import joblib
from wildfire.features.regions import build_region_season
cm = joblib.load(cfg.path_for('models') / 'count_model.joblib')
region = build_region_season(cfg, df)
latest = region[region['year'] == region['year'].max()].copy()
latest['pred'] = cm.predict(latest)
lo, hi = cm.predict_interval(latest)
latest['lo'], latest['hi'] = lo, hi
latest = latest.sort_values('pred')
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(latest['pred'], latest['region'],
            xerr=[latest['pred']-latest['lo'], latest['hi']-latest['pred']],
            fmt='o', color='#7a3b2e', ecolor='#ddd', capsize=3)
ax.scatter(latest['fire_count'], latest['region'], marker='|', s=200, color='k', label='actual')
ax.legend(); ax.set_xlabel('fires per season'); ax.set_title('Predicted vs actual (95% PI)')
plt.tight_layout()

**Next:** `04_maps_and_detection.ipynb` — the risk surface and the CNN detector.